In [1]:
import torch

print("CUDA disponible:", torch.cuda.is_available())
print("Número de GPUs:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU activa:", torch.cuda.get_device_name(0))
    print("Versión CUDA PyTorch:", torch.version.cuda)

: 

In [2]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.get_arch_list())


2.11.0.dev20251215+cu128
12.8
['sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']


In [3]:
import torch

device = torch.device("cuda")
x = torch.randn(4096, 4096, device=device)
y = torch.matmul(x, x)
torch.cuda.synchronize()

print("OK")


OK


In [4]:
import torch

print("CUDA available:", torch.cuda.is_available())


CUDA available: True


In [5]:
print("GPU count:", torch.cuda.device_count())
print("Current device:", torch.cuda.current_device())


GPU count: 1
Current device: 0


In [6]:
print("GPU name:", torch.cuda.get_device_name(0))
print("Compute capability:", torch.cuda.get_device_capability(0))


GPU name: NVIDIA GeForce RTX 5060 Ti
Compute capability: (12, 0)


In [7]:
print("PyTorch version:", torch.__version__)
print("CUDA runtime version:", torch.version.cuda)
print("Compiled arch list:", torch.cuda.get_arch_list())


PyTorch version: 2.11.0.dev20251215+cu128
CUDA runtime version: 12.8
Compiled arch list: ['sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']


In [8]:
device = torch.device("cuda")

x = torch.randn(4096, 4096, device=device)
y = torch.matmul(x, x)

torch.cuda.synchronize()
print("GPU computation successful")


GPU computation successful


In [9]:
import time
import numpy as np

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms


# -----------------------------
# Benchmark 1: Random Forest
# -----------------------------
def benchmark_random_forest():
    print("\n=== Benchmark 1: RandomForest (scikit-learn) ===")

    data = load_breast_cancer()
    X_train, X_test, y_train, y_test = train_test_split(
        data.data, data.target, test_size=0.2, random_state=42
    )

    model = RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        n_jobs=-1,
        random_state=42
    )

    start = time.perf_counter()
    model.fit(X_train, y_train)
    end = time.perf_counter()

    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    print(f"Tiempo entrenamiento RF: {end - start:.3f} segundos")
    print(f"Accuracy RF: {acc:.4f}")


# -----------------------------
# Benchmark 2: PyTorch Nightly NN
# -----------------------------
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(28 * 28, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.net(x)


def benchmark_pytorch(device):
    print("\n=== Benchmark 2: Neural Network (PyTorch Nightly) ===")
    print(f"Device: {device}")

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Lambda(lambda x: x.view(-1))
    ])

    train_dataset = datasets.MNIST(
        root="./data",
        train=True,
        download=True,
        transform=transform
    )

    test_dataset = datasets.MNIST(
        root="./data",
        train=False,
        download=True,
        transform=transform
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=128,
        shuffle=True,
        num_workers=4,
        pin_memory=(device.type == "cuda")
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=256,
        shuffle=False
    )

    model = MLP().to(device)

    # PyTorch 2.x / nightly compiler
    model = torch.compile(model)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    torch.cuda.synchronize() if device.type == "cuda" else None
    start = time.perf_counter()

    model.train()
    for epoch in range(25):
        for x, y in train_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            output = model(x)
            loss = criterion(output, y)
            loss.backward()
            optimizer.step()

    torch.cuda.synchronize() if device.type == "cuda" else None
    end = time.perf_counter()

    # Evaluation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(device)
            y = y.to(device)
            output = model(x)
            pred = output.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)

    acc = correct / total

    print(f"Tiempo entrenamiento NN: {end - start:.3f} segundos")
    print(f"Accuracy NN: {acc:.4f}")


# -----------------------------
# Main
# -----------------------------
if __name__ == "__main__":
    print("=== BENCHMARK CPU / ML (PyTorch Nightly) ===")
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")

    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")

    device = (
        torch.device("cuda") if torch.cuda.is_available()
        else torch.device("cpu")
    )

    benchmark_random_forest()
    benchmark_pytorch(device)


ModuleNotFoundError: No module named 'sklearn'